# ACT の学習(段階2の2個のデモ s2_mix_1025) — 積み木のお片付け(2026-10-29 版)
Drive の同期フォルダ(`PAI2026 Workspace`。自動で探す)にある段階2のデータ `s2_mix_1025`(425件。SmolVLA の3回目と同じ)で ACT を学習し、チェックポイントを **日付入りの新しい名前** で同じ Drive フォルダに保存する。

**使い方**: ランタイムのタイプを GPU(L4 または T4)にして、新しいランタイムで上から順に実行する(ランタイムを使い回した後なら、先に「ランタイム → 接続解除して削除」)。所要時間は、セル3が数分、セル6が約30分(L4)。

**この版での変更点**
- データ名 `NAME` と保存名 `RUN` を分けた。保存先が既にあれば止まる(上書きしない)。
- 学習後に `train_args.json` の steps と保存先を自動で照合する。

**10/29 版の変更点**: 10/21 版(`act_train.ipynb`)から、データ名 `NAME` を `s2_mix_1025` に、保存名 `RUN` を `s2_mix_1025_act30k_1029` に変えただけ。ステップ数(30,000)、バッチ(8)、その他の設定は同じ。元のノートブックは変更していない。


In [ ]:
# 1) Drive をつなぎ、同期フォルダを自動で探す(lerobot_dataset を含むフォルダ)
from google.colab import drive
drive.mount('/content/drive')
import os, glob, sys, shutil, subprocess, json, hashlib
def sh(cmd):
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end='')
    p.wait()
    assert p.returncode == 0, '失敗(終了コード %d): %s' % (p.returncode, cmd)

NAME  = 's2_mix_1025'                # 学習に使うデータ名(段階2、425件)
STEPS = 30000                        # 学習ステップ数
RUN   = 's2_mix_1025_act30k_1029'    # 保存名(日付入り。既存のチェックポイントには書かない)
assert RUN != NAME, 'RUN は NAME と別の名前にする'

DRIVE_ROOT = None                    # 自動で見つからないときだけ '/content/drive/MyDrive/…' を直接書く
if DRIVE_ROOT is None:
    hits = sorted(glob.glob('/content/drive/MyDrive/lerobot_dataset/'+NAME) + glob.glob('/content/drive/MyDrive/*/lerobot_dataset/'+NAME) + glob.glob('/content/drive/MyDrive/*/*/lerobot_dataset/'+NAME) + glob.glob('/content/drive/MyDrive/*/*/*/lerobot_dataset/'+NAME))
    print('見つかった候補:', hits)
    assert hits, 'lerobot_dataset/%s が Drive に見つからない。DRIVE_ROOT を直接書く' % NAME
    DRIVE_ROOT = os.path.dirname(os.path.dirname(hits[0]))
print('DRIVE_ROOT =', DRIVE_ROOT)
OUT = f'{DRIVE_ROOT}/checkpoints/train/{RUN}'
assert not os.path.exists(OUT), '保存先が既にある。RUN の名前を変える: ' + OUT
for p in ['lerobot_dataset/'+NAME, 'pai_tools/tools/pai_img/train_img.py', 'lerobot_src/pyproject.toml']:
    ok = os.path.exists(f'{DRIVE_ROOT}/{p}')
    print(('OK  ' if ok else '無い ') + p)
    assert ok, '必要なファイルが無い: ' + p
print('保存先(新規):', OUT)

In [ ]:
# 2) GPU の確認
sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
print(sys.version)

In [ ]:
# 3) lerobot と、ローカルに合わせた版の依存を入れる(数分)
shutil.rmtree('/content/lerobot_src', ignore_errors=True)
shutil.copytree(f'{DRIVE_ROOT}/lerobot_src', '/content/lerobot_src')
sh('pip install -q torch==2.7.0 torchvision==0.22.0 torchcodec==0.4.0')
sh('pip install -q --only-binary=:all: "av>=14.2.0" datasets==3.6.0')
sh('pip install -q -e /content/lerobot_src')
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU が使えない。ランタイムのタイプを GPU にする'
import importlib
for _p in ['/content/lerobot_src', '/content/lerobot_src/src']:   # 同じセッション内では editable install が見えないので、パスを足す
    if os.path.isdir(_p) and _p not in sys.path: sys.path.insert(0, _p)
importlib.invalidate_caches()
from lerobot.common.policies.act.modeling_act import ACTPolicy; print('lerobot import OK')

In [ ]:
# 4) データを Colab のローカルディスクへ(Drive のまま読むと遅い)
shutil.rmtree('/content/dataset', ignore_errors=True)
os.makedirs('/content/dataset')
shutil.copytree(f'{DRIVE_ROOT}/lerobot_dataset/{NAME}', f'/content/dataset/{NAME}')
shutil.copy(f'{DRIVE_ROOT}/pai_tools/tools/pai_img/train_img.py', '/content/train_img.py')
print(os.listdir(f'/content/dataset/{NAME}'))

In [ ]:
# 5) 動作確認(30ステップ、1分以内)。出力先は Colab のローカルで、Drive には書かない
sh(f'python /content/train_img.py --dataset-root /content/dataset/{NAME} --repo-id {NAME} --output /content/ckpt_test --steps 30 --batch-size 8 --num-workers 2')

In [ ]:
# 6) 本番の学習(約30分)。途中で止めない。チェックポイントは Drive の新しい保存先に直接保存する
assert not os.path.exists(OUT), '保存先が既にある: ' + OUT
sh(f'python /content/train_img.py --dataset-root /content/dataset/{NAME} --repo-id {NAME} --output "{OUT}" --steps {STEPS} --batch-size 8 --num-workers 2 --save-every 5000')

In [ ]:
# 7) 保存の確認: train_args.json の steps と保存先を照合し、ハッシュ(先頭16桁)を表示する
sh(f'ls -la "{OUT}"')
ta = json.load(open(f'{OUT}/train_args.json'))
print(json.dumps(ta, ensure_ascii=False, indent=1))
assert ta['steps'] == STEPS, 'steps が違う: %s' % ta['steps']
assert ta['output'] == OUT, 'output が違う: %s' % ta['output']
h = hashlib.sha256(open(f'{OUT}/model.safetensors','rb').read()).hexdigest()[:16]
print('model.safetensors sha16 =', h, ' size =', os.path.getsize(f'{OUT}/model.safetensors'))
print('確認 OK: steps=%d, 保存名=%s' % (ta['steps'], RUN))

## 終わったら
セル7の出力(`steps` と `sha16`)を作業ログに記録する。

**重要**: 平日用端末へは、Drive の `checkpoints/train/{RUN}` を、他のチェックポイントを上書きしない方法でコンテナに入れる(`s1_region` は書き換えない)。`sync_in.bat` は、手元の同名フォルダを Drive の内容で置き換えることがあるので、実行前に手順を確認する。

注意: Colab は、ブラウザのタブを閉じる、無操作が続く、使用時間の上限、のどれかで切れる。学習コードに再開機能はないので、途中で切れたら、最後に保存された 5000 ステップ単位のチェックポイントまでが残る。
